# Fine-Tune a Model Router using Azure OpenAI REST API

This notebook walks you through end-to-end fine-tuning of a **Model Router** using Azure OpenAI REST endpoints. The workflow follows the official [Azure OpenAI fine-tuning guide](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/fine-tuning?tabs=oai-sdk&pivots=rest-api#review-the-workflow-for-the-rest-api).

## Workflow
1. **Configure** – Set all parameters in a single config cell.
2. **Understand the data format** – Review how training data is structured.
3. **Upload data** – Upload training (and optionally test) files to Azure OpenAI.
4. **Submit fine-tuning job** – Kick off model training.
5. **Monitor job** – Poll until training completes.
6. **Download results** – Retrieve the training metrics CSV.
7. **Deploy the model** – Create a deployment for inference.
8. **Test the deployment** – Make a test call to verify the fine-tuned model.

## Prerequisites
- An Azure AI Foundry project with fine-tuning access.
- The **Azure AI Owner** role (required for deployment).
- Python packages: `requests`.
- An API key or Azure AD token for your resource.

## 0. Imports

All imports are consolidated here at the top of the notebook.

In [ ]:
import json
import os
import sys
import requests
import subprocess

# Ensure the src/ helper module is importable
sys.path.insert(0, os.path.abspath("."))

from src.finetuning_helpers import (
    upload_file,
    create_finetuning_job,
    poll_job_until_complete,
    list_job_events,
    download_result_file,
    deploy_finetuned_model,
)

## 1. Configuration

Set all user-configurable parameters below. This is the **only cell** you need to edit before running the notebook.

In [ ]:
# ── Azure AI Foundry Project Endpoint ─────────────────────────────────────
#AZURE_OPENAI_ENDPOINT = "https://<YOUR_RESOURCE_NAME>.services.ai.azure.com/api/projects/<YOUR_PROJECT_NAME>"


AZURE_RESOURCE_NAME = "alanmuoz-8961-resource"  # Replace with your actual resource name (without .openai.azure.com)
AZURE_PROJECT_NAME = "alanmuoz-8961"  
AZURE_OPENAI_ENDPOINT = f"https://{AZURE_RESOURCE_NAME}.services.ai.azure.com/api/projects/{AZURE_PROJECT_NAME}"  # Replace with your actual endpoint
AZURE_OPENAI_API_KEY  = "<YOUR_API_KEY>"  # Or set via environment variable

# ── Data Paths ─────────────────────────────────────────────────────────
TRAINING_FILE_PATH   = "data/zava_enterprise_train.jsonl"
VALIDATION_FILE_PATH = "data/zava_enterprise_test.jsonl"  # Optional – if not provided, training data is split 80:20 automatically

# ── Model ──────────────────────────────────────────────────────────────
BASE_MODEL = "model-router" 
SUFFIX     = "demo"      # Up to 18 chars; helps identify your fine-tuned model

# ── Deployment (used after training succeeds) ─────────────────────
SUBSCRIPTION_ID      = "381b38e9-9840-4719-a5a0-61d9585e1e91"
RESOURCE_GROUP       = "alanmuoz-rg-automldemo"
DEPLOYMENT_NAME      = "model-router-demo-4"  # Name for the deployed model
DEPLOYMENT_CAPACITY  = 10

# ── Monitoring ─────────────────────────────────────────────────────────
POLL_INTERVAL_SECONDS = 60  # How often to check job status

## 2. Understanding the Data Format

The training data must be in **JSONL** (JSON Lines) format. Each line is a JSON object with the following structure:

```json
{
  "messages": [
    {"role": "user", "content": "<your prompt>"}
  ],
  "metrics": {
    "model_a": 1,
    "model_b": 0
  },
  "usage": {
    "model_a": {"completion_tokens": 205, "prompt_tokens": 184},
    "model_b": {"completion_tokens": 179, "prompt_tokens": 184}
  }
}
```

### Field descriptions

| Field | Required | Description |
|-------|----------|-------------|
| `messages` | ✅ Yes | A list of chat messages following the Chat Completions API format. Typically contains a `user` message with the prompt. |
| `metrics` | ✅ Yes | A dictionary mapping each model name to a binary score (`1` = correct, `0` = incorrect). The model router uses these to learn which models perform well on which types of tasks. |
| `usage` | ✅ Yes | A dictionary mapping each model name to its token usage (`completion_tokens` and `prompt_tokens`). The router uses this to optimize for cost-efficiency alongside accuracy. |

### Notes
- **Validation data is optional.** If you do not provide a validation file, the training data is automatically split **80:20** (80% train, 20% validation).
- Files must be UTF-8 encoded and less than 512 MB.
- A minimum of 200 training examples is recommended for meaningful results.

Let's preview a sample from the training data:

In [3]:
# Preview the first training example
with open(TRAINING_FILE_PATH, "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())

print("=== Sample Training Record ===")
print(json.dumps(sample, indent=2)[:2000])  # Truncate for readability

=== Sample Training Record ===
{
  "messages": [
    {
      "role": "user",
      "content": "What's the current return rate for our cordless drill collection?"
    }
  ],
  "metrics": {
    "gpt-5_2025-08-07": 1,
    "gpt-5-mini_2025-08-07": 1,
    "gpt-5-nano_2025-08-07": 0
  },
  "usage": {
    "gpt-5_2025-08-07": {
      "prompt_tokens": 15,
      "completion_tokens": 1222
    },
    "gpt-5-mini_2025-08-07": {
      "prompt_tokens": 15,
      "completion_tokens": 702
    },
    "gpt-5-nano_2025-08-07": {
      "prompt_tokens": 15,
      "completion_tokens": 1349
    }
  }
}


## 3. Upload Data

Upload the training file (and optionally the validation file) to Azure OpenAI.

In [4]:
# Upload training file
print("Uploading training file...")
train_response = upload_file(AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, TRAINING_FILE_PATH)
training_file_id = train_response["id"]
print(f"Training file ID: {training_file_id}")

# Upload validation file (optional)
validation_file_id = None
if VALIDATION_FILE_PATH:
    print("Uploading validation file...")
    val_response = upload_file(AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, VALIDATION_FILE_PATH)
    validation_file_id = val_response["id"]
    print(f"Validation file ID: {validation_file_id}")
else:
    print("No validation file provided. Training data will be split 80:20 automatically.")

Uploading training file...
Training file ID: file-a85cd608f652416e9b2585b0716a2e77
Uploading validation file...
Validation file ID: file-10851f9f73cd4b78842c08cf1d6d79d3


## 4. Submit Fine-Tuning Job

Create a fine-tuning job with the uploaded data. Hyperparameters are optional — the service uses sensible defaults if not specified.

In [5]:
print("Submitting fine-tuning job...")
job_response = create_finetuning_job(
    endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    model=BASE_MODEL,
    training_file_id=training_file_id,
    validation_file_id=validation_file_id,
    seed=105,  # For reproducibility
)

job_id = job_response["id"]
print(f"Fine-tuning job submitted!")
print(f"Job ID: {job_id}")
print(json.dumps(job_response, indent=2))

Submitting fine-tuning job...
Fine-tuning job submitted!
Job ID: ftjob-05c422738b2d4c7ea63410f1efb95d13
{
  "hyperparameters": {
    "n_epochs": 1,
    "batch_size": 16,
    "learning_rate_multiplier": 1
  },
  "additionalParameters": {
    "completionOverride": "True"
  },
  "seed": 105,
  "method": {
    "type": "supervised",
    "supervised": {
      "hyperparameters": {
        "n_epochs": 1,
        "batch_size": 16,
        "learning_rate_multiplier": 1
      }
    }
  },
  "trainingType": "globalStandard",
  "status": "pending",
  "model": "model-router-2025-11-18",
  "metadata": {
    "base_model": "model-router",
    "model_version": "2025-11-18"
  },
  "training_file": "file-a85cd608f652416e9b2585b0716a2e77",
  "validation_file": "file-10851f9f73cd4b78842c08cf1d6d79d3",
  "estimated_finish": 1779121185,
  "id": "ftjob-05c422738b2d4c7ea63410f1efb95d13",
  "created_at": 1779118725,
  "object": "fine_tuning.job"
}


## 5. Monitor the Fine-Tuning Job

Poll the job status until it reaches a terminal state (`succeeded`, `failed`, or `cancelled`). This can take minutes to hours depending on dataset size and model.

In [6]:
# Poll until the job finishes
final_status = poll_job_until_complete(
    endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    job_id=job_id,
    poll_interval=POLL_INTERVAL_SECONDS,
)

print("\n=== Final Job Status ===")
print(json.dumps(final_status, indent=2))

Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pending
Job ftjob-05c422738b2d4c7ea63410f1efb95d13 status: pendi

In [7]:
# View training events / logs
events = list_job_events(AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, job_id)
print(json.dumps(events, indent=2))

{
  "has_more": false,
  "data": [
    {
      "id": "ftevent-a292d5f9aa814c26b4b79cdb6bd564cf",
      "created_at": 1779120988,
      "level": "info",
      "message": "Training tokens billed: 0",
      "type": "message",
      "object": "fine_tuning.job.event"
    },
    {
      "id": "ftevent-bb47a31a65df466086c23abdd8cf9d77",
      "created_at": 1779120988,
      "level": "info",
      "message": "Completed results file: file-e92e5f7215274ec1a5089346dbfc1a39",
      "type": "message",
      "object": "fine_tuning.job.event"
    },
    {
      "id": "ftevent-575034a4d8674bf78dc2578653fd7a53",
      "created_at": 1779120979,
      "level": "info",
      "message": "Job succeeded.",
      "type": "message",
      "object": "fine_tuning.job.event"
    },
    {
      "id": "ftevent-da7aac2a7d914087b191e4e343ea5352",
      "created_at": 1779120047,
      "level": "info",
      "message": "Created results file: file-e92e5f7215274ec1a5089346dbfc1a39",
      "type": "message",
      "object

## 6. Download Result File

After a successful job, Azure OpenAI generates a `results.csv` with training metrics (loss, accuracy per step/epoch). Let's download it.

In [ ]:
if final_status["status"] == "succeeded":
    result_file_id = final_status["result_files"][0]
    print(f"Result file ID: {result_file_id}")

    download_result_file(
        endpoint=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        file_id=result_file_id,
        output_path="results.csv",
    )

    # Preview the results
    import pandas as pd
    df = pd.read_csv("results.csv")
    print(df.head(10))
else:
    print(f"Job did not succeed. Status: {final_status['status']}")
    print("Check the events above for error details.")

Result file ID: file-e92e5f7215274ec1a5089346dbfc1a39
   base_accuracy  base_cost  ft_accuracy_quality_mode  ft_cost_quality_mode  \
0             89       0.56                        93                  2.17   

   ft_accuracy_balanced_mode  ft_cost_balanced_mode  ft_accuracy_cost_mode  \
0                         91                   0.37                     70   

   ft_cost_cost_mode  
0               0.03  


## 7. Deploy the Fine-Tuned Model

Deployment uses the **Azure Management REST API** (control plane), which requires an Azure AD bearer token rather than an API key.

Generate a token by running in your terminal:
```bash
az login
az account get-access-token --query accessToken -o tsv
```

> **Note:** You need the **Azure AI Owner** role to create deployments.


> ⚠️ **Important:** The **subset** feature is **not supported** for fine-tuned model deployments. All inference requests will only be routed to the models present in the training data.

In [ ]:
# Get the fine-tuned model name from the completed job
fine_tuned_model = final_status.get("fine_tuned_model")
print(f"Fine-tuned model: {fine_tuned_model}")

AZURE_AD_TOKEN = subprocess.check_output(["az", "account", "get-access-token", "--query", "accessToken", "-o", "tsv"]).decode().strip()

if fine_tuned_model:
    print(f"Deploying model '{fine_tuned_model}' as '{DEPLOYMENT_NAME}'...")
    deploy_response = deploy_finetuned_model(
        token=AZURE_AD_TOKEN,
        subscription_id=SUBSCRIPTION_ID,
        resource_group=RESOURCE_GROUP,
        resource_name=AZURE_RESOURCE_NAME,
        deployment_name=DEPLOYMENT_NAME,
        fine_tuned_model=fine_tuned_model,
        sku_name="GlobalStandard",
        sku_capacity=DEPLOYMENT_CAPACITY, #RPM
        mode="quality"
    )
    print("Deployment created successfully!")
    print(json.dumps(deploy_response, indent=2))
else:
    print("No fine-tuned model available. Ensure the training job succeeded.")

Fine-tuned model: model-router-2025-11-18.ft-05c422738b2d4c7ea63410f1efb95d13
Deploying model 'model-router-2025-11-18.ft-05c422738b2d4c7ea63410f1efb95d13' as 'model-router-demo-4'...
Deployment created successfully!
{
  "id": "/subscriptions/381b38e9-9840-4719-a5a0-61d9585e1e91/resourceGroups/alanmuoz-rg-automldemo/providers/Microsoft.CognitiveServices/accounts/alanmuoz-8961-resource/deployments/model-router-demo-4",
  "type": "Microsoft.CognitiveServices/accounts/deployments",
  "name": "model-router-demo-4",
  "sku": {
    "name": "GlobalStandard",
    "capacity": 10
  },
  "properties": {
    "model": {
      "format": "OpenAI",
      "name": "model-router-2025-11-18.ft-05c422738b2d4c7ea63410f1efb95d13",
      "version": "1",
      "sourceAccount": "/subscriptions/381b38e9-9840-4719-a5a0-61d9585e1e91/resourceGroups/alanmuoz-rg-automldemo/providers/Microsoft.CognitiveServices/accounts/alanmuoz-8961-resource"
    },
    "versionUpgradeOption": "NoAutoUpgrade",
    "currentCapacity": 

## 8. Test the Fine-Tuned Deployment

Make a single test call to the deployed fine-tuned model using the Chat Completions REST API to verify it is working correctly.

In [12]:
# Read the first example from the test file (or training file) to use as a test prompt
test_file = VALIDATION_FILE_PATH or TRAINING_FILE_PATH
with open(test_file, "r", encoding="utf-8") as f:
    test_sample = json.loads(f.readline())

test_messages = test_sample["messages"]
print("=== Test Prompt ===")
print(test_messages[0]["content"][:500])

# Call the deployed fine-tuned model
url = f"https://{AZURE_RESOURCE_NAME}.openai.azure.com/openai/deployments/{DEPLOYMENT_NAME}/chat/completions?api-version=2025-01-01-preview"

headers = {"Content-Type": "application/json", "api-key": AZURE_OPENAI_API_KEY}
payload = {"messages": test_messages}

response = requests.post(url, headers=headers, json=payload)
response.raise_for_status()
result = response.json()


print("\n=== Model Response ===")
print(result["choices"][0]["message"]["content"])
print(f"\nTokens used — prompt: {result['usage']['prompt_tokens']}, completion: {result['usage']['completion_tokens']}")

=== Test Prompt ===
Build a customer segmentation model using RFM analysis on our transaction data. Cluster customers into at least 5 segments, profile each by demographics and purchase behavior, and recommend targeted marketing strategies per segment.

=== Model Response ===
Below is a **complete end‑to‑end playbook** for building a customer‑segmentation model using **RFM (Recency‑Frequency‑Monetary) analysis**, clustering the RFM scores into **≥ 5 distinct segments**, enriching those clusters with **demographic data**, profiling each segment, and finally turning the insight into **targeted marketing tactics**.  

You can follow the steps in the order presented, adapt the code to your own data warehouse, and automate the pipeline for quarterly (or monthly) refreshes.

---  

## 1️⃣  What you need before you start  

| Item | Typical source | Example columns |
|------|----------------|-----------------|
| **Transactional data** | Order / sales table (e.g., `orders`, `transactions`) | `